In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Both sources belong to the same development pool; mentor test data are separate.
DATASET_ROOTS = [PROJECT_ROOT / "dataset", PROJECT_ROOT / "dataset_extra"]
for root in DATASET_ROOTS:
    if not root.is_dir():
        raise FileNotFoundError(f"Missing dataset source: {root}")
DATASET_ROOT = DATASET_ROOTS[0]  # Only for the introductory single-image example.
print("Dataset sources:", [root.name for root in DATASET_ROOTS])

In [ ]:
for DATASET_ROOT in DATASET_ROOTS:
    print("Source:", DATASET_ROOT.name)
    for split_name in ["train", "test", "unclean"]:
        split_path = DATASET_ROOT / split_name
        if not split_path.exists():
            continue
    
        print(f"{split_name}: exists={split_path.is_dir()}")

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
class_counts = []
for DATASET_ROOT in DATASET_ROOTS:
    print("Source:", DATASET_ROOT.name)
    for split_name in ["train", "test", "unclean"]:
        split_path = DATASET_ROOT / split_name
        if not split_path.exists():
            continue
        total_images = 0
    
        print(f"\nSplit: {split_name}")
    
        for class_path in sorted(split_path.iterdir()):
            if not class_path.is_dir():
                continue
    
            image_paths = [
                path
                for path in class_path.iterdir()
                if path.is_file()
                and path.suffix.lower() in IMAGE_EXTENSIONS
            ]
    
            count = len(image_paths)
            total_images += count
    
            class_counts.append({
                "split": split_name,
                "class_name": class_path.name,
                "count": count,
            })
    
            print(f"  {class_path.name}: {count}")
    
        print(f"Total: {total_images}")

In [ ]:
from PIL import Image

sample_path = next(
    path
    for path in sorted((DATASET_ROOT / "train" / "ambulance").iterdir())
    if path.is_file()
    and path.suffix.lower() in IMAGE_EXTENSIONS
)

with Image.open(sample_path) as image:
    image.load()

    print("File:", sample_path.name)
    print("Format:", image.format)
    print("Size:", image.size)
    print("Color mode:", image.mode)

In [ ]:
def inspect_image(image_path):
    result = {
        "path": str(image_path),
        "width": None,
        "height": None,
        "format": None,
        "mode": None,
        "readable": False,
        "error": None,
    }

    try:
        with Image.open(image_path) as image:
            image.load()

            result["width"] = image.width
            result["height"] = image.height
            result["format"] = image.format
            result["mode"] = image.mode
            result["readable"] = True

    except (OSError, ValueError) as error:
        result["error"] = str(error)

    return result

In [ ]:
sample_info = inspect_image(sample_path)
print(sample_info)

In [ ]:
image_records = []
for DATASET_ROOT in DATASET_ROOTS:
    print("Source:", DATASET_ROOT.name)
    for split_name in ["train", "test", "unclean"]:
        split_path = DATASET_ROOT / split_name
        if not split_path.exists():
            continue
    
        for class_path in sorted(split_path.iterdir()):
            if not class_path.is_dir():
                continue
    
            for image_path in sorted(class_path.iterdir()):
                if not image_path.is_file():
                    continue
    
                if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                    continue
    
                record = inspect_image(image_path)
    
                record["source_dataset"] = DATASET_ROOT.name
                record["source_split"] = split_name
                record["label"] = class_path.name
    
                image_records.append(record)
    

print("Inspected images:", len(image_records))
unreadable_records = [r for r in image_records if not r["readable"]]
print("Unreadable images:", len(unreadable_records))

In [ ]:
readable_records = [
    record
    for record in image_records
    if record["readable"]
]

for record in readable_records:
    record["short_side"] = min(record["width"], record["height"])
    record["aspect_ratio"] = record["width"] / record["height"]

for split_name in ["train", "test", "unclean"]:
    split_records = [
        record
        for record in readable_records
        if record["source_split"] == split_name
    ]

    if not split_records:
        continue

    widths = [record["width"] for record in split_records]
    heights = [record["height"] for record in split_records]
    ratios = [record["aspect_ratio"] for record in split_records]

    print(f"\nSplit: {split_name}")
    print(f"Width range: {min(widths)} to {max(widths)}")
    print(f"Height range: {min(heights)} to {max(heights)}")
    print(f"Aspect ratio range: {min(ratios):.2f} to {max(ratios):.2f}")

In [ ]:
from IPython.display import display

train_records = [
    record
    for record in readable_records
    if record["source_split"] == "train"
]

smallest_records = sorted(
    train_records,
    key=lambda record: record["short_side"],
)[:6]

smallest_paths = {
    record["path"]
    for record in smallest_records
}

remaining_records = [
    record
    for record in train_records
    if record["path"] not in smallest_paths
]

most_elongated_records = sorted(
    remaining_records,
    key=lambda record: max(
        record["aspect_ratio"],
        1 / record["aspect_ratio"],
    ),
    reverse=True,
)[:6]

for title, records in [
    ("Smallest images", smallest_records),
    ("Most elongated images", most_elongated_records),
]:
    print(f"\n{title}")

    for record in records:
        print(
            f"Class: {record['label']} | "
            f"Size: {record['width']}x{record['height']} | "
            f"Ratio: {record['aspect_ratio']:.2f}"
        )
        print("Path:", record["path"])

        with Image.open(record["path"]) as image:
            preview = image.convert("RGB")
            preview.thumbnail((320, 320))
            display(preview)

In [ ]:
import hashlib

def compute_image_hash(image_path):
    with Image.open(image_path) as image:
        rgb_image = image.convert("RGB")

        hasher = hashlib.sha256()
        hasher.update(str(rgb_image.size).encode("utf-8"))
        hasher.update(rgb_image.tobytes())

        return hasher.hexdigest()

In [ ]:
sample_hash = compute_image_hash(sample_path)
print(sample_hash)

In [ ]:
hash_groups = {}

for record in readable_records:
    image_hash = compute_image_hash(record["path"])
    record["content_hash"] = image_hash

    if image_hash not in hash_groups:
        hash_groups[image_hash] = []

    hash_groups[image_hash].append(record)

duplicate_groups = [
    group
    for group in hash_groups.values()
    if len(group) > 1
]

print("Images checked:", len(readable_records))
print("Unique image contents:", len(hash_groups))
print("Duplicate groups:", len(duplicate_groups))


extra_copies = sum(
    len(group) - 1
    for group in duplicate_groups
)

print("Extra duplicate copies:", extra_copies)

In [ ]:
cross_split_groups = []
label_conflict_groups = []

for group_number, group in enumerate(duplicate_groups, start=1):
    splits = {record["source_split"] for record in group}
    labels = {record["label"] for record in group}

    if len(splits) > 1:
        cross_split_groups.append(group)

    if len(labels) > 1:
        label_conflict_groups.append(group)

    print(f"\nGroup {group_number}")

    for record in group:
        print(
            f"  {record['source_split']}/{record['label']}"
            f" ﷿﷿﷿ {record['path']}"
        )

    print("Across splits:", len(splits) > 1)
    print("Label conflict:", len(labels) > 1)

print("\nDuplicate groups across splits:", len(cross_split_groups))
print("Groups with conflicting labels:", len(label_conflict_groups))

In [ ]:
for group in label_conflict_groups:
    print("Same image, different labels:")

    for record in group:
        print(
            f"  split={record['source_split']}, "
            f"label={record['label']}, "
            f"path={record['path']}"
        )

    with Image.open(group[0]["path"]) as image:
        preview = image.convert("RGB")
        preview.thumbnail((400, 400))
        display(preview)

In [ ]:
import csv
from collections import Counter

known_labels = sorted({r["label"] for r in readable_records if r["source_split"] == "train"})
class_to_idx = {label: i for i, label in enumerate(known_labels)}
review_file = PROJECT_ROOT / "reports" / "data_review_decisions.csv"
previous_decisions = {}
if review_file.exists():
    with review_file.open(newline="", encoding="utf-8") as file:
        previous_decisions = {(r["path"], r["content_hash"]): r for r in csv.DictReader(file)}

# Clean labelled train and former test are eligible. Unclean requires an explicit decision.
# No automatic merge of neysan into vanet and no automatic ninth class.
base_records = []
pending_records = []
conflict_contents = set()
for content_hash, group in hash_groups.items():
    processed = []
    for original in group:
        record = dict(original)
        relative = Path(record["path"]).relative_to(PROJECT_ROOT).as_posix()
        decision = previous_decisions.get((relative, content_hash), {})
        action = decision.get("decision", "pending")
        if action not in {"pending", "accept", "exclude", "relabel"}:
            raise ValueError(f"Invalid review decision: {relative}")
        if action == "exclude":
            continue
        if action == "relabel":
            if decision.get("reviewed_label") not in known_labels:
                raise ValueError(f"Relabel requires a known reviewed_label: {relative}")
            record["label"] = decision["reviewed_label"]
        record["approved"] = (
            record["label"] in known_labels and
            (action in {"accept", "relabel"} or record["source_split"] in {"train", "test"})
        )
        processed.append(record)
    if len({r["label"] for r in processed}) > 1:
        conflict_contents.add(content_hash)
        pending_records.extend(processed)
        continue
    approved = [r for r in processed if r["approved"]]
    if approved:
        # One representative per pixel content, before splitting.
        base_records.append(min(approved, key=lambda r: r["path"]))
    else:
        pending_records.extend(processed)

base_records.sort(key=lambda r: r["path"])
unclean_known_candidates = [r for r in pending_records if r["source_split"] == "unclean" and r["label"] in known_labels]
neysan_candidates = [r for r in pending_records if r["label"] == "neysan"]
print("Eligible unique images:", len(base_records))
print("Unclean awaiting review:", len(unclean_known_candidates))
print("Neysan awaiting label policy/review:", len(neysan_candidates))
print("Unresolved conflicting contents:", len(conflict_contents))

In [ ]:
from collections import Counter

candidate_counts = Counter(r["label"] for r in unclean_known_candidates)
labelled_counts = Counter(r["label"] for r in base_records)
for class_name in known_labels:
    print(f"{class_name:10} | eligible unique: {labelled_counts[class_name]} | unclean awaiting review: {candidate_counts[class_name]}")
print("Total unclean known-class candidates:", len(unclean_known_candidates))


In [ ]:
import json
import random

SPLIT_SEED = 42
VALIDATION_RATIO = 0.20
split_file = PROJECT_ROOT / "reports" / "base_split.json"

# Keep the current validation set fixed when reviewed unclean images join training.
if split_file.exists():
    prior_split = json.loads(split_file.read_text(encoding="utf-8"))
    prior_labels = {row["path"]: row for row in prior_split["samples"]}
    current_records = {
        Path(row["path"]).relative_to(PROJECT_ROOT).as_posix(): row
        for row in base_records
    }
    validation_paths = prior_split["validation_paths"]
    missing = set(validation_paths) - set(current_records)
    if missing:
        raise ValueError(
            f"Frozen validation has {len(missing)} missing or conflicting images; "
            "resolve labels before creating a new split."
        )
    for path in validation_paths:
        old = prior_labels[path]
        current = current_records[path]
        if old["label"] != current["label"] or old["content_hash"] != current["content_hash"]:
            raise ValueError(f"Frozen validation image changed label or content: {path}")
    validation_records = [current_records[path] for path in validation_paths]
    validation_path_set = set(validation_paths)
    training_records = [
        row for path, row in current_records.items() if path not in validation_path_set
    ]
    validation_frozen = True
else:
    # First run: make a reproducible stratified 80/20 split.
    random_generator = random.Random(SPLIT_SEED)
    training_records = []
    validation_records = []
    for class_name in known_labels:
        class_records = [r for r in base_records if r["label"] == class_name]
        if len(class_records) < 2:
            raise ValueError(f"Not enough images for train/validation: {class_name}")
        class_records.sort(key=lambda r: r["path"])
        random_generator.shuffle(class_records)
        n_validation = min(
            len(class_records) - 1,
            max(1, round(len(class_records) * VALIDATION_RATIO)),
        )
        validation_records.extend(class_records[:n_validation])
        training_records.extend(class_records[n_validation:])
    validation_frozen = False

from collections import Counter
train_counts = Counter(row["label"] for row in training_records)
val_counts = Counter(row["label"] for row in validation_records)
for class_name in known_labels:
    print(f"{class_name}: train={train_counts[class_name]}, validation={val_counts[class_name]}")
print("Validation frozen from existing manifest:", validation_frozen)
print("Total train:", len(training_records))
print("Total validation:", len(validation_records))


In [ ]:
training_hashes = {r["content_hash"] for r in training_records}
validation_hashes = {r["content_hash"] for r in validation_records}
assert len(training_hashes) == len(training_records)
assert len(validation_hashes) == len(validation_records)
assert not training_hashes & validation_hashes
assert len(training_records) + len(validation_records) == len(base_records)
print("Exact-content train/validation overlap:", len(training_hashes & validation_hashes))
print("Mentor-held test data are not available to this notebook.")

In [ ]:
import json
import hashlib

split_file = PROJECT_ROOT / "reports" / "base_split.json"
split_file.parent.mkdir(parents=True, exist_ok=True)

def relative_paths(records):
    return sorted(Path(r["path"]).relative_to(PROJECT_ROOT).as_posix() for r in records)

split_manifest = {
    "seed": SPLIT_SEED,
    "validation_ratio": VALIDATION_RATIO,
    "sources": [root.name for root in DATASET_ROOTS],
    "class_to_idx": class_to_idx,
    "samples": [
        {"path": Path(r["path"]).relative_to(PROJECT_ROOT).as_posix(),
         "label": r["label"], "content_hash": r["content_hash"]}
        for r in base_records
    ],
    "training_paths": relative_paths(training_records),
    "validation_paths": relative_paths(validation_records),
}
split_manifest["fingerprint"] = hashlib.sha256(
    json.dumps(split_manifest, sort_keys=True).encode("utf-8")
).hexdigest()
split_file.write_text(json.dumps(split_manifest, indent=2)+"\n", encoding="utf-8")
print("Saved split:", split_file.name)
print("Training / validation:", len(training_records), len(validation_records))

# Only aggregate counts belong in the publishable report.
summary = {
    "source_counts": {root.name: sum(r["source_dataset"] == root.name for r in image_records) for root in DATASET_ROOTS},
    "readable_images": len(readable_records), "unique_contents": len(hash_groups),
    "duplicate_groups": len(duplicate_groups), "unresolved_conflicting_contents": len(conflict_contents),
    "eligible_unique_images": len(base_records), "training_images": len(training_records),
    "validation_images": len(validation_records), "unclean_pending": len(unclean_known_candidates),
    "neysan_pending": len(neysan_candidates), "seed": SPLIT_SEED,
    "validation_ratio": VALIDATION_RATIO,
    "validation_frozen": validation_frozen,
    "unclean_training_images": sum(r["source_split"] == "unclean" for r in training_records),
}
(PROJECT_ROOT / "reports" / "data_summary.json").write_text(json.dumps(summary, indent=2)+"\n")
print(summary)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

review_folders = ["train/vanet", "unclean/vanet", "unclean/neysan", "train/kamyun", "train/kamyunet"]
IMAGES_PER_PAGE = 25
for DATASET_ROOT in DATASET_ROOTS:
    for relative_folder in review_folders:
        image_paths = sorted(
            (DATASET_ROOT / relative_folder).glob("*.jpg")
        )
        print(f"{relative_folder}: {len(image_paths)} images")
    
        for start in range(0, len(image_paths), IMAGES_PER_PAGE):
            page_paths = image_paths[start:start + IMAGES_PER_PAGE]
            fig, axes = plt.subplots(5, 5, figsize=(16, 17))
            axes = axes.ravel()
    
            for offset, image_path in enumerate(page_paths):
                ax = axes[offset]
                with Image.open(image_path) as image:
                    ax.imshow(image.convert("RGB"))
                ax.set_title(
                    f"{start + offset + 1}: {image_path.name}",
                    fontsize=10,
                )
    
            for ax in axes:
                ax.axis("off")
    
            fig.suptitle(
                f"{relative_folder} | "
                f"{start + 1}-{start + len(page_paths)} / {len(image_paths)}"
            )
            plt.tight_layout(rect=(0, 0, 1, 0.97))
            plt.show()
            plt.close(fig)

## سیاست مشترک آماده‌سازی داده و بررسی برچسب‌ها

هر دو منبع در یک ممیزی بررسی می‌شوند. train و test قدیمیِ مجاز، به همراه unclean تأییدشده، یک مخزن مشترک می‌سازند. ابتدا محتوای تکراری یک نماینده می‌گیرد؛ در نخستین اجرا تقسیم تصادفی ۸۰/۲۰ به تفکیک کلاس با seed=42 ساخته می‌شود. پس از آن validation ثابت می‌ماند و نمونه‌های تأییدشدهٔ unclean فقط به آموزش اضافه می‌شوند تا مقایسهٔ آزمایش‌ها منصفانه باشد. تست نهایی نزد منتورهاست. این تقسیم تازه است و نتایج قبلی CNN را نمی‌توان نتیجهٔ آن دانست.

موارد unclean تا ثبت accept، exclude یا relabel در data_review_decisions.csv وارد تقسیم‌بندی نمی‌شوند. برای relabel باید reviewed_label از هشت کلاس باشد. یک محتوای دارای دو برچسب متفاوت تا رفع تعارض کنار گذاشته می‌شود. هیچ فایل خامی پاک یا تغییر برچسب داده نمی‌شود؛ تصمیم‌ها فقط در فهرست آموزشی اثر دارند.

neysan به‌طور خودکار به vanet یا کلاس نهم تبدیل نمی‌شود. وجود خودروهای شبیه نیسان در vanet یعنی تفاوت نام برچسب الزاماً به معنی ظاهر کاملاً جدید نیست. confidenceِ softmax تضمینی برای تشخیص اشیای ناشناخته نیست. معیار نیاز به بازبینی باید با داده‌های محلی سنجیده شود.

در مرور قبلیِ پوشه‌های وانت منبع اول، مدل‌های مختلف وانت و خودروهایی با ظاهر مشابه نیسان دیده شدند؛ تصاویر منبع دوم هنوز به‌صورت تک‌به‌تک تأیید نشده‌اند. سلول تصویری برای مرور هر دو منبع است؛ نمایش تصاویر به معنی تأیید خودکار برچسب نیست.

base_split.json تنها تقسیم فعال و data_summary.json تنها خلاصهٔ ممیزی فعال است. گزارش موردیِ بازبینی، داده‌های خام، وزن‌ها و خروجی‌های تصویری به دلیل NDA فقط محلی‌اند و وارد گیت نمی‌شوند. hash فقط تکرار دقیق را بررسی می‌کند؛ تصاویر مشابه و فریم‌های یک خودرو نیاز به بررسی جداگانه دارند.